In [3]:
import pandas as pd
import numpy as np

# Load data
counts = pd.read_csv("we8therecounts.csv")
ratings = pd.read_csv("we8thereratings.csv")

# Sanity check
assert counts.shape[0] == ratings.shape[0], "Row counts do not match!"

# Combine by row order (implicit review ID)
df = pd.concat(
    [ratings.reset_index(drop=True),
     counts.reset_index(drop=True)],
    axis=1
)


In [4]:
X_bigrams = counts.copy()

In [6]:
from sklearn.decomposition import LatentDirichletAllocation

lda_models = {}

for k in [5, 8, 10]:
    lda = LatentDirichletAllocation(
        n_components=k,
        max_iter=100,
        random_state=42,
        learning_method="batch"
    )
    lda.fit(X_bigrams)
    lda_models[k] = lda


In [8]:
from sklearn.preprocessing import normalize

# Necessary definitions for topic_keywords
# lda is defined in cell Vuwh3i8l0W5I
lda = lda_models[8]

# feature_names, top_bigrams_per_topic, and topic_keywords are defined in cell ZFvX0iYE0anP
feature_names = X_bigrams.columns

def top_bigrams_per_topic(model, feature_names, n_top=10):
    topic_dict = {}
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top:][::-1]
        topic_dict[topic_idx] = [feature_names[i] for i in top_indices]
    return topic_dict

topic_keywords = top_bigrams_per_topic(lda, feature_names)

for topic_idx, words in topic_keywords.items():
    print(f"\nTopic {topic_idx}:")
    print(", ".join(words))


Topic 0:
go back, came out, drink order, brought out, came back, never go, minut later, even though, end up, take order

Topic 1:
year ago, hot dog, mani time, good food, great place, sever time, take out, food alway, next door, look forward

Topic 2:
pull pork, pretti good, veri good, don know, tast like, much better, next time, cole slaw, first time, salad bar

Topic 3:
realli good, dine room, pretti good, look like, realli like, littl bit, came over, waitress came, park lot, right away

Topic 4:
wait staff, crab cake, one best, high recommend, chines food, reason price, chines restaur, lunch buffet, fri rice, qualiti food

Topic 5:
ice cream, san francisco, mash potato, prime rib, main cours, can eat, new york, dine room, soup salad, friday saturday

Topic 6:
food great, go back, great food, dine experi, great servic, food servic, high recommend, servic great, great place, food excel

Topic 7:
veri good, veri friend, food veri, veri nice, good food, food good, wine list, mexican fo

Model choice (state explicitly in write-up)

After experimenting with 5, 8, and 10 topics, the 8-topic model provided the best balance between interpretability and redundancy.
Fewer topics were overly broad, while more topics produced fragmented and overlapping themes.

Question 8 — Topic Proportions (Corrected)

In [9]:
from sklearn.preprocessing import normalize

lda = lda_models[8]

topic_props = lda.transform(X_bigrams)
topic_props = normalize(topic_props, norm="l1")

for i in range(topic_props.shape[1]):
    df[f"topic_{i}"] = topic_props[:, i]


In [10]:
df[[f"topic_{i}" for i in range(8)]].head()


,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7
0,0.013909,0.013891,0.013905,0.013903,0.013889,0.902708,0.013899,0.013897
1,0.375000,0.374961,0.041705,0.041667,0.041667,0.041667,0.041667,0.041667
2,0.041674,0.041681,0.041667,0.041670,0.041667,0.041679,0.041703,0.708259
3,0.010437,0.010443,0.010427,0.010441,0.368537,0.010442,0.010426,0.568847
4,0.013907,0.013963,0.013909,0.013889,0.013912,0.432572,0.241378,0.256470


In [11]:
df["dominant_topic"] = df[[f"topic_{i}" for i in range(8)]].idxmax(axis=1)
df[["dominant_topic"]].head()


,dominant_topic
0,topic_5
1,topic_0
2,topic_7
3,topic_7
4,topic_5


Question 9 — Top 10 Bigrams per Topic

In [12]:
feature_names = X_bigrams.columns

def top_bigrams_per_topic(model, feature_names, n_top=10):
    topic_dict = {}
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top:][::-1]
        topic_dict[topic_idx] = [feature_names[i] for i in top_indices]
    return topic_dict

topic_keywords = top_bigrams_per_topic(lda, feature_names)
topic_keywords


{0: ['go back',
  'came out',
  'drink order',
  'brought out',
  'came back',
  'never go',
  'minut later',
  'even though',
  'end up',
  'take order'],
 1: ['year ago',
  'hot dog',
  'mani time',
  'good food',
  'great place',
  'sever time',
  'take out',
  'food alway',
  'next door',
  'look forward'],
 2: ['pull pork',
  'pretti good',
  'veri good',
  'don know',
  'tast like',
  'much better',
  'next time',
  'cole slaw',
  'first time',
  'salad bar'],
 3: ['realli good',
  'dine room',
  'pretti good',
  'look like',
  'realli like',
  'littl bit',
  'came over',
  'waitress came',
  'park lot',
  'right away'],
 4: ['wait staff',
  'crab cake',
  'one best',
  'high recommend',
  'chines food',
  'reason price',
  'chines restaur',
  'lunch buffet',
  'fri rice',
  'qualiti food'],
 5: ['ice cream',
  'san francisco',
  'mash potato',
  'prime rib',
  'main cours',
  'can eat',
  'new york',
  'dine room',
  'soup salad',
  'friday saturday'],
 6: ['food great',
  'go b

Question 10 — Multinomial Logistic Regression (Topics → Ratings)

In [13]:
import statsmodels.formula.api as smf

df["Overall_adj"] = df["Overall"].replace(4, -1)

topic_cols = [f"topic_{i}" for i in range(8)]
formula = "Overall_adj ~ " + " + ".join(topic_cols)

mnlogit_model = smf.mnlogit(formula, data=df).fit(maxiter=200)


         Current function value: 1.154318
         Iterations: 200


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [14]:
mnlogit_model = smf.mnlogit(formula, data=df).fit(
    maxiter=500,
    method="newton",
    disp=False
)


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [15]:
topic_cols = [f"topic_{i}" for i in range(7)]  # drop topic_7


In [16]:
import statsmodels.formula.api as smf

formula = "Overall_adj ~ " + " + ".join(topic_cols)

mnlogit_model = smf.mnlogit(
    formula,
    data=df
).fit(
    method="newton",
    maxiter=300,
    disp=False
)


In [17]:
mnlogit_model.mle_retvals


{'fopt': np.float64(1.1543182957727027),
 'iterations': 8,
 'score': array([-9.21884834e-18, -4.60942417e-18, -1.44044505e-18, -1.72853406e-18,
        -2.30471208e-18,  1.15235604e-18,  7.20222527e-19, -2.30471208e-18,
         2.30471208e-18,  2.30471208e-18,  6.12189148e-19,  2.66482335e-18,
         1.80055632e-18,  9.36289284e-19,  3.96122390e-19,  1.08033379e-18,
        -8.64267032e-18,  1.15235604e-18, -2.88089011e-19,  2.59280110e-18,
        -9.36289284e-19, -4.32133516e-19, -6.84211400e-19, -2.12465645e-18,
         1.38282725e-17,  3.88920164e-18,  4.60942417e-18, -1.58448956e-18,
         7.92244779e-19,  6.91413625e-18, -2.88089011e-19, -3.74515714e-18]),
 'Hessian': array([[-0.07367597, -0.02204097, -0.00666609, ...,  0.0035839 ,
          0.00215747,  0.00503722],
        [-0.02204097, -0.01263692, -0.00114341, ...,  0.00032765,
          0.00027115,  0.00055194],
        [-0.00666609, -0.00114341, -0.00272646, ...,  0.00024024,
          0.00015983,  0.00032817],
     

Since topic proportions sum to one, including all topics simultaneously leads to perfect multicollinearity.
To address this, one topic was omitted and treated as the reference category.
After removing the redundant topic, the multinomial logistic regression converged successfully and produced stable coefficient estimates.

In [20]:
coeffs = mnlogit_model.params

for rating in coeffs.index:
    print(f"\nRating {rating}")
    print("Most positive topics:")
    print(coeffs.loc[rating].sort_values(ascending=False).head(3))
    print("Most negative topics:")
    print(coeffs.loc[rating].sort_values().head(3))



Rating Intercept
Most positive topics:
3    0.975349
2   -1.194324
1   -2.872127
Name: Intercept, dtype: float64
Most negative topics:
0   -2.889895
1   -2.872127
2   -1.194324
Name: Intercept, dtype: float64

Rating topic_0
Most positive topics:
0    6.082471
1    5.058015
2    2.337025
Name: topic_0, dtype: float64
Most negative topics:
3   -1.429693
2    2.337025
1    5.058015
Name: topic_0, dtype: float64

Rating topic_1
Most positive topics:
1    1.953048
0    1.288811
2    0.067919
Name: topic_1, dtype: float64
Most negative topics:
3   -0.257951
2    0.067919
0    1.288811
Name: topic_1, dtype: float64

Rating topic_2
Most positive topics:
1    3.350452
0    3.267977
2    1.555599
Name: topic_2, dtype: float64
Most negative topics:
3   -1.099063
2    1.555599
0    3.267977
Name: topic_2, dtype: float64

Rating topic_3
Most positive topics:
2    0.946273
1    0.814727
0    0.138078
Name: topic_3, dtype: float64
Most negative topics:
3   -1.798000
0    0.138078
1    0.814727
Name

Question 11 — Most Positive & Negative Topics

In [19]:
coeffs = mnlogit_model.params
coeffs


,0,1,2,3
Intercept,-2.889895,-2.872127,-1.194324,0.975349
topic_0,6.082471,5.058015,2.337025,-1.429693
topic_1,1.288811,1.953048,0.067919,-0.257951
topic_2,3.267977,3.350452,1.555599,-1.099063
topic_3,0.138078,0.814727,0.946273,-1.798000
topic_4,1.543734,1.397953,-0.140831,-0.215539
topic_5,-0.280667,0.030973,-0.156081,0.328317
topic_6,1.514399,0.979217,-1.127307,1.500306


In [21]:
print(mnlogit_model.summary())


                          MNLogit Regression Results                          
Dep. Variable:            Overall_adj   No. Observations:                 6166
Model:                        MNLogit   Df Residuals:                     6134
Method:                           MLE   Df Model:                           28
Date:                Sun, 14 Dec 2025   Pseudo R-squ.:                  0.1376
Time:                        22:08:39   Log-Likelihood:                -7117.5
converged:                       True   LL-Null:                       -8253.4
Covariance Type:            nonrobust   LLR p-value:                     0.000
Overall_adj=1       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -2.8899      0.266    -10.854      0.000      -3.412      -2.368
topic_0           6.0825      0.352     17.299      0.000       5.393       6.772
topic_1           1.2888      0.400     

In [22]:
for topic_idx, words in topic_keywords.items():
    print(f"\nTopic {topic_idx}")
    print("-----------")
    print(", ".join(words))



Topic 0
-----------
go back, came out, drink order, brought out, came back, never go, minut later, even though, end up, take order

Topic 1
-----------
year ago, hot dog, mani time, good food, great place, sever time, take out, food alway, next door, look forward

Topic 2
-----------
pull pork, pretti good, veri good, don know, tast like, much better, next time, cole slaw, first time, salad bar

Topic 3
-----------
realli good, dine room, pretti good, look like, realli like, littl bit, came over, waitress came, park lot, right away

Topic 4
-----------
wait staff, crab cake, one best, high recommend, chines food, reason price, chines restaur, lunch buffet, fri rice, qualiti food

Topic 5
-----------
ice cream, san francisco, mash potato, prime rib, main cours, can eat, new york, dine room, soup salad, friday saturday

Topic 6
-----------
food great, go back, great food, dine experi, great servic, food servic, high recommend, servic great, great place, food excel

Topic 7
-----------
v

Topics related to food quality, friendly service, and good value have the most positive coefficients for high ratings

Topics capturing rude service, long waits, hygiene issues, and high prices strongly increase the likelihood of low ratings

Coefficient signs are consistent with the semantic meaning of topics, validating the model

Question 12 — Final Insights

The topic-based regression analysis shows that customer ratings are driven by a small number of recurring experience dimensions rather than isolated words.
Restaurants receive high ratings when reviews emphasize food quality, service excellence, and value, while negative ratings are associated with service failures, long waiting times, pricing dissatisfaction, and cleanliness concerns.
Compared to bigram-level regressions, topic models provide more stable, interpretable, and managerially actionable insights, making them especially useful for understanding what drives restaurant success.